This notebook is for analysing the NHS monthly admission data to understand the distribution

In [93]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
# Get list of all CSV files in data dir
files = [file for file in os.listdir('../data') if file.endswith('.csv')]

# Get list of dataframes from all CSV files
dfs = [pd.read_csv(f'../data/raw/{file}') for file in files]

for i, df in enumerate(dfs):
    print(f"Dataframe {i} shape: {df.shape}") # All dataframes have 22 columns and similar rows


# Merge all dataframes into one
df = pd.concat(dfs, ignore_index=True)

Dataframe 0 shape: (199, 22)
Dataframe 1 shape: (202, 22)
Dataframe 2 shape: (202, 22)
Dataframe 3 shape: (201, 22)
Dataframe 4 shape: (200, 22)
Dataframe 5 shape: (198, 22)
Dataframe 6 shape: (202, 22)
Dataframe 7 shape: (198, 22)
Dataframe 8 shape: (202, 22)
Dataframe 9 shape: (198, 22)
Dataframe 10 shape: (197, 22)
Dataframe 11 shape: (198, 22)


In [95]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2397 entries, 0 to 2396
Data columns (total 22 columns):
 #   Column                                                      Non-Null Count  Dtype
---  ------                                                      --------------  -----
 0   Period                                                      2397 non-null   str  
 1   Org Code                                                    2397 non-null   str  
 2   Parent Org                                                  2397 non-null   str  
 3   Org name                                                    2397 non-null   str  
 4   A&E attendances Type 1                                      2397 non-null   int64
 5   A&E attendances Type 2                                      2397 non-null   int64
 6   A&E attendances Other A&E Department                        2397 non-null   int64
 7   A&E attendances Booked Appointments Type 1                  2397 non-null   int64
 8   A&E attendances Booked Appoin

None

## Cleaning

### Date Parsing

In [96]:
# Looking at the df information, the period column needs to be parsed as a date. 

# Checking the unique values in the period column 
display(df['Period'].unique())

# Remove the total rows, where any row has total.
df = df[
    ~df["Period"].str.contains("total", case=False, na=False) &
    ~df["Org Code"].str.contains("total", case=False, na=False)
]


<StringArray>
[    'MSitAE-APRIL-2025',                 'TOTAL',      'MSitAE-JULY-2025',
 'MSitAE-SEPTEMBER-2025',       'MSitAE-MAY-2025',     'MSitAE-MARCH-2026',
  'MSitAE-DECEMBER-2025',                'TOTAL ',    'MSitAE-AUGUST-2025',
  'MSitAE-NOVEMBER-2025',      'MSitAE-JUNE-2025',  'MSitAE-FEBRUARY-2026',
   'MSitAE-OCTOBER-2025',   'MSitAE-JANUARY-2026',                 'Total']
Length: 15, dtype: str

In [97]:

df['Period'] = df['Period'].str.replace('MSitAE-', '') # Remove the MSitAE- prefix from the period column
df['Period'] = pd.to_datetime(df['Period'], format='%B-%Y') # Parse the values, like 'APRIL-2025', as dates

### Whitespace removal

In [98]:
# Removing the whitespace from all rows
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

### Column name cleaning

In [99]:
display(df.columns)

# remove the special chars from colum names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace('&', '', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

# for all column names, replace more than one underscore with a single underscore
df.columns = (
    df.columns
      .str.replace(r'_+', '_', regex=True)
)
display(df.columns)

Index(['Period', 'Org Code', 'Parent Org', 'Org name',
       'A&E attendances Type 1', 'A&E attendances Type 2',
       'A&E attendances Other A&E Department',
       'A&E attendances Booked Appointments Type 1',
       'A&E attendances Booked Appointments Type 2',
       'A&E attendances Booked Appointments Other Department',
       'Attendances over 4hrs Type 1', 'Attendances over 4hrs Type 2',
       'Attendances over 4hrs Other Department',
       'Attendances over 4hrs Booked Appointments Type 1',
       'Attendances over 4hrs Booked Appointments Type 2',
       'Attendances over 4hrs Booked Appointments Other Department',
       'Patients who have waited 4-12 hs from DTA to admission',
       'Patients who have waited 12+ hrs from DTA to admission',
       'Emergency admissions via A&E - Type 1',
       'Emergency admissions via A&E - Type 2',
       'Emergency admissions via A&E - Other A&E department',
       'Other emergency admissions'],
      dtype='str')

Index(['period', 'org_code', 'parent_org', 'org_name', 'ae_attendances_type_1',
       'ae_attendances_type_2', 'ae_attendances_other_ae_department',
       'ae_attendances_booked_appointments_type_1',
       'ae_attendances_booked_appointments_type_2',
       'ae_attendances_booked_appointments_other_department',
       'attendances_over_4hrs_type_1', 'attendances_over_4hrs_type_2',
       'attendances_over_4hrs_other_department',
       'attendances_over_4hrs_booked_appointments_type_1',
       'attendances_over_4hrs_booked_appointments_type_2',
       'attendances_over_4hrs_booked_appointments_other_department',
       'patients_who_have_waited_4_12_hs_from_dta_to_admission',
       'patients_who_have_waited_12+_hrs_from_dta_to_admission',
       'emergency_admissions_via_ae_type_1',
       'emergency_admissions_via_ae_type_2',
       'emergency_admissions_via_ae_other_ae_department',
       'other_emergency_admissions'],
      dtype='str')

## Analysis

### Totals

In [ ]:
# Type 1 = Consultant-led major A&E departments
# Type 2 = Consultant-led single-speciality A&E departments
# Typ 3 / Other = Other A&E departments, like minor injury, walk ins

# A&E Attendnaces = All attendances
# A&E attendances < / > 4 hours from arrival to addmission, transfer, or discharge

# Emergency addmissions via A&E = All emergency admissions


df['total_ae_attendances'] = (
    df['ae_attendances_type_1'] + df['ae_attendances_type_2'] + df['ae_attendances_other_ae_department']
)

df['total_over_4hrs'] = (
    df['attendances_over_4hrs_type_1'] +
    df['attendances_over_4hrs_type_2'] +
    df['attendances_over_4hrs_other_department']
)

df['total_emergency_admissions'] = (
    df['emergency_admissions_via_ae_type_1'] +
    df['emergency_admissions_via_ae_type_2'] +
    df['emergency_admissions_via_ae_other_ae_department'] +
    df['other_emergency_admissions']
)

# Percentage of attendances that resulted in an emergency admission
df['admission_rate'] = df['total_emergency_admissions'] / df['total_ae_attendances'] * 100

df = df[df['total_ae_attendances'] > 0].copy() # Remove rows where total attendances is 0 

### Grouping

In [101]:
monthly_totals = df.groupby('period', as_index=False)[
    ['total_ae_attendances', 'total_over_4hrs', 'total_emergency_admissions']
].sum().sort_values('period')
monthly_totals['admission_rate'] = monthly_totals['total_emergency_admissions'] / monthly_totals['total_ae_attendances'] * 100

top_orgs = (
    df.groupby('org_name', as_index=False)[
        ['total_ae_attendances', 'total_over_4hrs', 'total_emergency_admissions']
    ].sum()
    .sort_values('total_ae_attendances', ascending=False)
    .head(10)
)
top_orgs['admission_rate'] = top_orgs['total_emergency_admissions'] / top_orgs['total_ae_attendances'] * 100


regional_totals = (
    df.groupby('parent_org', as_index=False)[
        ['total_ae_attendances', 'total_over_4hrs', 'total_emergency_admissions']
    ].sum()
    .sort_values('total_ae_attendances', ascending=False)
)
regional_totals['admission_rate'] = regional_totals['total_emergency_admissions'] / regional_totals['total_ae_attendances'] * 100

In [104]:
display(monthly_totals)
display(top_orgs)
display(regional_totals)

,period,total_ae_attendances,total_over_4hrs,total_emergency_admissions,admission_rate
0,2025-04-01,2209687,571127,526829,23.841793
1,2025-05-01,2309298,581375,543550,23.537456
2,2025-06-01,2268166,566734,534524,23.566353
3,2025-07-01,2320374,560285,557918,24.044314
4,2025-08-01,2182723,539040,524973,24.051288
5,2025-09-01,2227502,567723,534054,23.975467
6,2025-10-01,2314177,610640,552621,23.879807
7,2025-11-01,2261926,595846,530318,23.445418
8,2025-12-01,2240325,599421,540624,24.131499
9,2026-01-01,2235959,629487,544576,24.355366


,org_name,total_ae_attendances,total_over_4hrs,total_emergency_admissions,admission_rate
8,BARTS HEALTH NHS TRUST,534304,153878,66662,12.476418
130,ROYAL FREE LONDON NHS FOUNDATION TRUST,507130,114606,85065,16.773806
86,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,479173,131207,78966,16.479643
171,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,450266,169154,214477,47.633399
108,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,425375,127633,120425,28.310314
47,FRIMLEY HEALTH NHS FOUNDATION TRUST,424503,115018,79744,18.785262
93,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,414213,120791,118676,28.650960
180,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,361528,127607,125821,34.802560
6,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",360604,80732,87301,24.209659
75,LEEDS TEACHING HOSPITALS NHS TRUST,358590,86667,58378,16.279874


,parent_org,total_ae_attendances,total_over_4hrs,total_emergency_admissions,admission_rate
2,NHS ENGLAND MIDLANDS,4899893,1376024,1255265,25.618212
1,NHS ENGLAND LONDON,4844022,1114064,752929,15.543468
3,NHS ENGLAND NORTH EAST AND YORKSHIRE,4192609,1044524,1127194,26.885264
5,NHS ENGLAND SOUTH EAST,4109133,955323,996954,24.261906
4,NHS ENGLAND NORTH WEST,3790594,1061287,923696,24.368107
0,NHS ENGLAND EAST OF ENGLAND,2752778,701624,660219,23.983736
6,NHS ENGLAND SOUTH WEST,2375997,659996,717793,30.210181


## Averages

Baseline numbers for the syntheitc data generation, targeting one department

In [ ]:
baseline_int = {
    'avg_ae_attendances': int(round(df['total_ae_attendances'].mean())),
    'avg_over_4hrs': int(round(df['total_over_4hrs'].mean())),
    'avg_emergency_admissions': int(round(df['total_emergency_admissions'].mean())),
    'avg_admission_rate': round(df['admission_rate'].mean(), 2)
}

baseline_int

{'avg_ae_attendances': 12006,
 'avg_over_4hrs': 3078,
 'avg_emergency_admissions': 2865,
 'avg_admission_rate': np.float64(18.02)}